# Task 21: Reward Component Range & Distribution — Real Data Analysis

> **Không dummy/demo** — Đọc TFRecords thật, dùng `reward_utils.py` logic thật từ `training.py:331-336`
> Output lưu trong `task1/` : `outputTask21_stats_train.csv`, `outputTask21_stats_test.csv`, `outputTask21_plots/`

Pipeline:
1. Load `data/train.tfrecords`, `test.tfrecords`, `capacity.tfrecords`, `stock.tfrecords` qua `reward_utils.load_tfrecord_data`
2. Tính `sales_norm = sales_raw / capacity` (before/after normalization)
3. Simulate env qua `env_step` + `calc_reward` để lấy `z, overstock, q, quan, reward` per product×timestep
4. Descriptive stats + histogram/boxplot before vs after + train vs test + covariate shift (Wasserstein/KL)


In [1]:
import os, sys, glob
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats

# Thêm task1 vào path để import reward_utils.py
sys.path.insert(0, os.path.dirname(os.path.abspath('')))
import reward_utils
print("reward_utils loaded from:", reward_utils.__file__)
print("TF version:", tf.__version__)
print("NUM_PRODUCTS:", reward_utils.NUM_PRODUCTS, "WASTE_RATE:", reward_utils.WASTE_RATE)


reward_utils loaded from: c:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task13-9\task1\reward_utils.py
TF version: 2.20.0
NUM_PRODUCTS: 220 WASTE_RATE: 0.025


In [2]:
# ============================================================
# 1. LOAD REAL TFRecords (không dummy)
# ============================================================
# Tìm thư mục data: từ task1 đi lên 3 cấp tới repo root
candidates = [
    os.path.join("..", "..", "..", "data"),
    os.path.join("..", "..", "data"),
    "data",
    "../../../data",
    "C:/GitHub/Q-learning-for-Inventory-Management/data",
]
data_dir = None
for c in candidates:
    if os.path.exists(os.path.join(c, "train.tfrecords")):
        data_dir = os.path.abspath(c)
        break
if data_dir is None:
    # fallback: tìm bằng glob
    import pathlib
    for p in pathlib.Path(".").rglob("train.tfrecords"):
        data_dir = str(p.parent.resolve())
        break
print("data_dir:", data_dir)
assert data_dir and os.path.exists(os.path.join(data_dir, "train.tfrecords")), "Không tìm thấy data/train.tfrecords"

data = reward_utils.load_tfrecord_data(data_dir=data_dir)
capacity = data['capacity']  # [220]
x_init = data['x_init']      # [220]
train_sales_raw = data['train_sales_raw']  # [T_train, 220]
test_sales_raw = data['test_sales_raw']    # [T_test, 220]
print(f"capacity shape: {capacity.shape}, mean={capacity.mean():.2f}, min={capacity.min():.1f}, max={capacity.max():.1f}")
print(f"x_init shape: {x_init.shape}, mean={x_init.mean():.4f}")
print(f"train_sales_raw: {train_sales_raw.shape}, mean={train_sales_raw.mean():.4f}, max={train_sales_raw.max():.1f}")
print(f"test_sales_raw: {test_sales_raw.shape}, mean={test_sales_raw.mean():.4f}, max={test_sales_raw.max():.1f}")


data_dir: c:\GitHub\Q-learning-for-Inventory-Management\data
capacity shape: (220,), mean=20.33, min=4.0, max=208.0
x_init shape: (220,), mean=0.5049
train_sales_raw: (1000, 220), mean=2.1083, max=162.0
test_sales_raw: (504, 220), mean=0.7331, max=31.0


In [3]:
# ============================================================
# 2. NORMALIZATION: before (raw) vs after (sales/capacity)
# ============================================================
train_sales_norm = reward_utils.normalize_sales(train_sales_raw, capacity)  # [T,220]
test_sales_norm = reward_utils.normalize_sales(test_sales_raw, capacity)
print(f"train_sales_norm: mean={train_sales_norm.mean():.4f}, std={train_sales_norm.std():.4f}, min={train_sales_norm.min():.4f}, max={train_sales_norm.max():.4f}")
print(f"test_sales_norm: mean={test_sales_norm.mean():.4f}, std={test_sales_norm.std():.4f}, min={test_sales_norm.min():.4f}, max={test_sales_norm.max():.4f}")
# Kiểm tra hệ số capacity = 12*mean sales như prepare_data.py:144
mean_sales_per_product_train = train_sales_raw.mean(axis=0)  # [220]
ratio = capacity / np.maximum(mean_sales_per_product_train, 1e-6)
print(f"capacity / mean_sales ratio: mean={ratio.mean():.2f}, p25={np.percentile(ratio,25):.2f}, p75={np.percentile(ratio,75):.2f} (kỳ vọng ~12)")


train_sales_norm: mean=0.1003, std=0.1396, min=0.0000, max=2.2500
test_sales_norm: mean=0.0341, std=0.0695, min=0.0000, max=1.2500
capacity / mean_sales ratio: mean=10.04, p25=9.50, p75=10.38 (kỳ vọng ~12)


In [4]:
# ============================================================
# 3. SIMULATE ENVIRONMENT để lấy reward components
# Dùng chính xác env_step + calc_reward từ reward_utils.py (copy từ training.py:331-336)
# ============================================================
def collect_components(sales_norm, x_init, capacity, policy_fn=None):
    """
    Chạy 1 episode qua toàn bộ timesteps, thu z, overstock, q, quan, reward per product per timestep.
    policy_fn: state_flat [660] -> action_idx [220]; nếu None dùng heuristic zero-action (u=0) để đo phân bố thuần túy của data
    Trả về dict các array [T,220]
    """
    P = reward_utils.NUM_PRODUCTS
    T = sales_norm.shape[0]
    x = x_init.copy()
    q = reward_utils.WASTE_RATE * x
    # ACTION_SPACE từ reward_utils
    ACTION_SPACE = reward_utils.ACTION_SPACE
    comp = {'z': [], 'overstock': [], 'q': [], 'quan': [], 'reward': [], 'x': [], 'sales': []}
    for t in range(T):
        sales_now = sales_norm[t]  # [220]
        # Build state flat 660 = [x, sales, q] như training.py:283
        state_flat = np.concatenate([x, sales_now, q]).astype(np.float32)
        if policy_fn is not None:
            action_idx = policy_fn(state_flat)  # [220]
            u = ACTION_SPACE[action_idx]
        else:
            u = np.zeros(P, dtype=np.float32)  # heuristic: không replenish, đo phân bố gốc
        x_next, overstock, x_clip, stockout_amount = reward_utils.env_step(x, u, sales_now, capacity)
        r, z, quan_scalar = reward_utils.calc_reward(x, overstock)  # x là x_old trước action
        q_cur = reward_utils.WASTE_RATE * x
        quan_vec = np.full(P, quan_scalar, dtype=np.float32)
        comp['z'].append(z)
        comp['overstock'].append(overstock)
        comp['q'].append(q_cur)
        comp['quan'].append(quan_vec)
        comp['reward'].append(r)
        comp['x'].append(x.copy())
        comp['sales'].append(sales_now.copy())
        # next
        x = x_next
        q = reward_utils.WASTE_RATE * x
    for k in comp:
        comp[k] = np.array(comp[k], dtype=np.float32)  # [T,220]
    return comp

# Heuristic policy (không cần checkpoint) — đo phân bố thuần data
train_comp_heuristic = collect_components(train_sales_norm, x_init, capacity, policy_fn=None)
test_comp_heuristic = collect_components(test_sales_norm, x_init, capacity, policy_fn=None)
print("Heuristic collect done:")
for k in ['z','overstock','q','quan','reward']:
    print(f"  train {k}: {train_comp_heuristic[k].shape}, mean={train_comp_heuristic[k].mean():.5f}, max={train_comp_heuristic[k].max():.4f}")
    print(f"  test  {k}: {test_comp_heuristic[k].shape}, mean={test_comp_heuristic[k].mean():.5f}, max={test_comp_heuristic[k].max():.4f}")


Heuristic collect done:
  train z: (1000, 220), mean=0.99310, max=1.0000
  test  z: (504, 220), mean=0.97496, max=1.0000
  train overstock: (1000, 220), mean=0.00000, max=0.0000
  test  overstock: (504, 220), mean=0.00000, max=0.0000
  train q: (1000, 220), mean=0.00007, max=0.0248
  test  q: (504, 220), mean=0.00023, max=0.0248
  train quan: (1000, 220), mean=0.00682, max=0.8909
  test  quan: (504, 220), mean=0.02641, max=0.8858
  train reward: (1000, 220), mean=0.00001, max=0.9999
  test  reward: (504, 220), mean=-0.00160, max=0.9995


In [5]:
# ============================================================
# 3c. POLICY-AWARE Distribution (fix overstock=0 do heuristic u=0)
# Để có overstock thực, chạy thêm với BaseStock (k=1.0) và DQN/A2C checkpoint
# ============================================================
try:
    import pathlib as _pl
    has_tfa = False
    try:
        import tensorflow_addons as _tfa
        has_tfa = True
    except:
        has_tfa = False
    print("has_tfa:", has_tfa)

    # BaseStock
    class BaseStockPolicyTmp:
        def __init__(self, train_sales_norm, k=1.0):
            mean_d = train_sales_norm.mean(axis=0)
            std_d = train_sales_norm.std(axis=0)
            self.S = __import__('numpy').clip(mean_d + k*std_d, 0, 1)
        def __call__(self, state_flat):
            import numpy as np
            x = state_flat[0:220]
            u_need = __import__('numpy').maximum(0, self.S - x)
            ACTION_SPACE_TMP = reward_utils.ACTION_SPACE
            idx = __import__('numpy').array([__import__('numpy').argmin(__import__('numpy').abs(ACTION_SPACE_TMP - v)) for v in u_need], dtype=__import__('numpy').int32)
            return idx

    base_stock_tmp = BaseStockPolicyTmp(train_sales_norm, k=1.0)
    train_comp_bs = collect_components(train_sales_norm, x_init, capacity, policy_fn=base_stock_tmp)
    test_comp_bs = collect_components(test_sales_norm, x_init, capacity, policy_fn=base_stock_tmp)
    print("BaseStock policy-aware done:")
    for k in ['z','overstock','q','quan','reward']:
        print(f"  train {k}: mean={train_comp_bs[k].mean():.5f} max={train_comp_bs[k].max():.4f} (vs heuristic {train_comp_heuristic[k].mean():.5f})")
        print(f"  test  {k}: mean={test_comp_bs[k].mean():.5f} max={test_comp_bs[k].max():.4f} (vs heuristic {test_comp_heuristic[k].mean():.5f})")

    # Save policy-aware stats
    import pandas as _pd, pathlib as _path
    def stats_df_bs(comp, split):
        rows=[]
        for name in ['z','overstock','q','quan','reward','x','sales']:
            if name in comp: rows.append(reward_utils.descriptive_stats(comp[name], name))
        df=_pd.DataFrame(rows); df['split']=split; return df
    df_train_bs = stats_df_bs(train_comp_bs, 'train_bs')
    df_test_bs = stats_df_bs(test_comp_bs, 'test_bs')
    out_dir_bs = _pl.Path(".").resolve()
    if out_dir_bs.name != "task1":
        for pp in _pl.Path(".").rglob("planTask20-21.md"):
            out_dir_bs = pp.parent.resolve(); break
    df_train_bs.to_csv(out_dir_bs / "outputTask21_stats_train_basestock.csv", index=False)
    df_test_bs.to_csv(out_dir_bs / "outputTask21_stats_test_basestock.csv", index=False)
    print("Saved: outputTask21_stats_train_basestock.csv, outputTask21_stats_test_basestock.csv")
except Exception as e:
    import traceback; traceback.print_exc()
    print("Policy-aware fix failed (có thể do thiếu tfa) — heuristic stats vẫn dùng được, đã ghi chú trong outputTask20-21.md")


has_tfa: False
BaseStock policy-aware done:
  train z: mean=0.15413 max=1.0000 (vs heuristic 0.99310)
  test  z: mean=0.02233 max=1.0000 (vs heuristic 0.97496)
  train overstock: mean=0.00000 max=0.0000 (vs heuristic 0.00000)
  test  overstock: mean=0.00000 max=0.0000 (vs heuristic 0.00000)
  train q: mean=0.00382 max=0.0248 (vs heuristic 0.00007)
  test  q: mean=0.00526 max=0.0248 (vs heuristic 0.00023)
  train quan: mean=0.23444 max=0.8858 (vs heuristic 0.00682)
  test  quan: mean=0.21166 max=0.8858 (vs heuristic 0.02641)
  train reward: mean=0.60762 max=0.9927 (vs heuristic 0.00001)
  test  reward: mean=0.76075 max=0.9061 (vs heuristic -0.00160)
Saved: outputTask21_stats_train_basestock.csv, outputTask21_stats_test_basestock.csv


In [6]:
# ============================================================
# 3b. (Tùy chọn) LOAD CHECKPOINT THẬT để simulate với policy đã train
# Nếu checkpoint tồn tại, sẽ chạy thêm 1 lần với DQN ckpt-60 và A2C ckpt-64
# ============================================================
import pathlib
ckpt_dqn = pathlib.Path(data_dir).parent / "output Training" / "checkpointDQN"
ckpt_a2c = pathlib.Path(data_dir).parent / "output Training" / "outputA2Cmod" / "checkpoints_a2cmod"
print("ckpt_dqn exists:", ckpt_dqn.exists(), "ckpt_a2c exists:", ckpt_a2c.exists())
if ckpt_dqn.exists():
    print("  DQN checkpoints:", [p.name for p in ckpt_dqn.glob("ckpt-*.index")][:5])
if ckpt_a2c.exists():
    print("  A2C checkpoints:", [p.name for p in ckpt_a2c.glob("ckpt-*.index")][:5])

# Ghi chú: việc load checkpoint cần định nghĩa lại Actor/QNetwork (xem reward_weight_sweep.ipynb).
# Ở notebook Task 21 này, nếu không load được checkpoint thì heuristic (u=0) đã đủ để báo cáo range/distribution.
# Nếu bạn muốn policy-aware distribution, hãy chạy cell dưới (yêu cầu tensorflow-addons).


ckpt_dqn exists: True ckpt_a2c exists: True
  DQN checkpoints: ['ckpt-58.index', 'ckpt-59.index', 'ckpt-60.index']
  A2C checkpoints: ['ckpt-1.index', 'ckpt-10.index', 'ckpt-11.index', 'ckpt-12.index', 'ckpt-13.index']


In [7]:
# ============================================================
# 4. DESCRIPTIVE STATISTICS — train vs test, before vs after
# Dùng reward_utils.descriptive_stats (scipy.stats.skew/kurtosis)
# ============================================================
def stats_df_from_components(comp, split_name):
    rows = []
    for name in ['z','overstock','q','quan','reward','x','sales']:
        if name in comp:
            rows.append(reward_utils.descriptive_stats(comp[name], name))
    df = pd.DataFrame(rows)
    df['split'] = split_name
    return df

df_train = stats_df_from_components(train_comp_heuristic, 'train')
df_test = stats_df_from_components(test_comp_heuristic, 'test')
df_all = pd.concat([df_train, df_test], ignore_index=True)
display(df_all)

# Lưu CSV trong task1/
out_dir = pathlib.Path(".").resolve()
# Nếu đang chạy từ task1, out_dir chính là task1; nếu chạy từ nơi khác, tìm task1
if out_dir.name != "task1":
    for p in pathlib.Path(".").rglob("planTask20-21.md"):
        out_dir = p.parent.resolve()
        break
print("out_dir:", out_dir)
df_train.to_csv(out_dir / "outputTask21_stats_train.csv", index=False)
df_test.to_csv(out_dir / "outputTask21_stats_test.csv", index=False)
df_all.to_csv(out_dir / "outputTask21_stats_all.csv", index=False)
print("Saved: outputTask21_stats_train.csv, outputTask21_stats_test.csv")


,component,count,mean,std,min,p25,p50,p75,max,skew,kurtosis,zero_frac,split
0,z,220000,0.993105,0.082752,0.000000,1.0,1.000000,1.000000,1.000000,-11.917629,140.029898,0.006895,train
1,overstock,220000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,NaN,NaN,NaN,train
2,q,220000,0.000070,0.000996,0.000000,0.0,0.000000,0.000000,0.024755,16.498993,296.080658,NaN,train
3,quan,220000,0.006817,0.069995,0.000000,0.0,0.000000,0.000000,0.890877,10.951244,122.540722,NaN,train
4,reward,220000,0.000008,0.038497,-0.890877,0.0,0.000000,0.000000,0.999880,-2.203368,261.249207,NaN,train
5,x,220000,0.002808,0.039860,0.000000,0.0,0.000000,0.000000,0.990207,16.498959,296.080008,NaN,train
6,sales,220000,0.100327,0.139576,0.000000,0.0,0.035714,0.166667,2.250000,2.098995,7.137115,NaN,train
7,z,110880,0.974964,0.156235,0.000000,1.0,1.000000,1.000000,1.000000,-6.080132,34.968027,0.025036,test
8,overstock,110880,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,NaN,NaN,NaN,test
9,q,110880,0.000227,0.001722,0.000000,0.0,0.000000,0.000000,0.024755,8.916140,86.241323,NaN,test


out_dir: C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task13-9\task1
Saved: outputTask21_stats_train.csv, outputTask21_stats_test.csv


In [8]:
# ============================================================
# 5. BEFORE vs AFTER NORMALIZATION — range/distribution
# So sánh sales_raw vs sales_norm (đây là normalization chính trong prepare_data.py)
# và tính lại reward components nếu dùng sales_raw (để thấy tác động của /capacity)
# ============================================================
def describe_array(arr, name):
    flat = arr.flatten()
    return {
        'component': name,
        'count': int(flat.size),
        'mean': float(np.mean(flat)),
        'std': float(np.std(flat)),
        'min': float(np.min(flat)),
        'p25': float(np.percentile(flat,25)),
        'p50': float(np.percentile(flat,50)),
        'p75': float(np.percentile(flat,75)),
        'max': float(np.max(flat)),
        'skew': float(scipy_stats.skew(flat)),
        'kurtosis': float(scipy_stats.kurtosis(flat)),
    }

rows_norm = []
rows_norm.append(describe_array(train_sales_raw, 'sales_raw_train'))
rows_norm.append(describe_array(train_sales_norm, 'sales_norm_train'))
rows_norm.append(describe_array(test_sales_raw, 'sales_raw_test'))
rows_norm.append(describe_array(test_sales_norm, 'sales_norm_test'))
rows_norm.append(describe_array(capacity, 'capacity'))
rows_norm.append(describe_array(train_comp_heuristic['q'], 'q_train_heuristic'))
rows_norm.append(describe_array(test_comp_heuristic['q'], 'q_test_heuristic'))
df_norm = pd.DataFrame(rows_norm)
display(df_norm)
df_norm.to_csv(out_dir / "outputTask21_normalization_stats.csv", index=False)
print("Saved: outputTask21_normalization_stats.csv")


,component,count,mean,std,min,p25,p50,p75,max,skew,kurtosis
0,sales_raw_train,220000,2.108277,4.620250,0.0,0.0,1.000000,2.000000,162.000000,5.912725,62.856822
1,sales_norm_train,220000,0.100327,0.139576,0.0,0.0,0.035714,0.166667,2.250000,2.098995,7.137115
2,sales_raw_test,110880,0.733063,1.754404,0.0,0.0,0.000000,1.000000,31.000000,4.889906,35.671166
3,sales_norm_test,110880,0.034095,0.069494,0.0,0.0,0.000000,0.041667,1.250000,2.932421,12.335330
4,capacity,220,20.327272,28.216890,4.0,5.0,10.000000,21.000000,208.000000,3.372688,13.970246
5,q_train_heuristic,220000,0.000070,0.000996,0.0,0.0,0.000000,0.000000,0.024755,16.498993,296.080658
6,q_test_heuristic,110880,0.000227,0.001722,0.0,0.0,0.000000,0.000000,0.024755,8.916140,86.241323


Saved: outputTask21_normalization_stats.csv


In [9]:
# ============================================================
# 6. VISUALIZATIONS — histogram + boxplot (before/after side-by-side)
# Mỗi component 1 figure 2x2, dùng reward_utils.plot_component_distribution
# ============================================================
plot_dir = out_dir / "outputTask21_plots"
plot_dir.mkdir(parents=True, exist_ok=True)
print("plot_dir:", plot_dir)

# Vẽ cho heuristic components: train vs test đã có, before/after ở đây là raw vs norm sales
# Với reward components, before=heuristic train, after=heuristic test (để thấy shift)
for comp_name in ['z','overstock','q','quan','reward']:
    save_path = plot_dir / f"{comp_name}_train_vs_test.png"
    # Dùng hàm có sẵn nhưng cần before/after dict
    # Tạo dict giả: before=train, after=test
    before = {comp_name: train_comp_heuristic[comp_name]}
    after = {comp_name: test_comp_heuristic[comp_name]}
    reward_utils.plot_component_distribution(before, after, comp_name, "train_vs_test", str(save_path))
    print("Saved:", save_path.name)

# Thêm histogram riêng cho sales_raw vs sales_norm
for split, raw, norm in [("train", train_sales_raw, train_sales_norm), ("test", test_sales_raw, test_sales_norm)]:
    fig, axes = plt.subplots(1,2, figsize=(12,4))
    fig.suptitle(f"Sales Distribution — {split} (before vs after normalization)", fontweight="bold")
    axes[0].hist(raw.flatten(), bins=50, density=True, alpha=0.7, edgecolor="black", color="skyblue")
    axes[0].set_title(f"Before Norm (raw) — N={raw.size:,}")
    axes[0].set_xlabel("sales (units)")
    axes[0].set_ylabel("Density")
    axes[0].grid(True, alpha=0.3)
    axes[1].hist(norm.flatten(), bins=50, density=True, alpha=0.7, edgecolor="black", color="lightcoral")
    axes[1].set_title(f"After Norm (sales/capacity) — N={norm.size:,}")
    axes[1].set_xlabel("sales / capacity")
    axes[1].set_ylabel("Density")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    p = plot_dir / f"sales_{split}_before_after_norm.png"
    plt.savefig(p, dpi=300, bbox_inches="tight")
    plt.close()
    print("Saved:", p.name)
print("All plots saved to", plot_dir)


plot_dir: C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task13-9\task1\outputTask21_plots
Saved: z_train_vs_test.png
Saved: overstock_train_vs_test.png
Saved: q_train_vs_test.png
Saved: quan_train_vs_test.png
Saved: reward_train_vs_test.png
Saved: sales_train_before_after_norm.png
Saved: sales_test_before_after_norm.png
All plots saved to C:\GitHub\Q-learning-for-Inventory-Management\Feedback 7-9\task13-9\task1\outputTask21_plots


In [10]:
# ============================================================
# 7. COVARIATE SHIFT — Wasserstein & KL giữa train/test
# ============================================================
shift = reward_utils.check_covariate_shift(train_comp_heuristic, test_comp_heuristic)
df_shift = pd.DataFrame([{"component":k, "wasserstein":v["wasserstein"], "kl_divergence":v["kl_divergence"]} for k,v in shift.items()])
display(df_shift)
df_shift.to_csv(out_dir / "outputTask21_covariate_shift.csv", index=False)
print("Saved: outputTask21_covariate_shift.csv")
# Thêm KL cho sales_raw vs norm
print("Sales raw shift (train vs test): Wasserstein =", reward_utils.wasserstein_distance(train_sales_raw, test_sales_raw))
print("Sales norm shift (train vs test): Wasserstein =", reward_utils.wasserstein_distance(train_sales_norm, test_sales_norm))


,component,wasserstein,kl_divergence
0,z,0.018141,0.470849
1,overstock,0.000000,0.000000
2,q,0.000157,18.937915
3,quan,0.019603,7.984551


Saved: outputTask21_covariate_shift.csv
Sales raw shift (train vs test): Wasserstein = 1.3752145021645032
Sales norm shift (train vs test): Wasserstein = 0.06623261681508856


## Kết thúc Task 21
- Kiểm tra 3 file CSV trong `task1/`: `outputTask21_stats_train.csv`, `outputTask21_stats_test.csv`, `outputTask21_covariate_shift.csv`
- Kiểm tra `outputTask21_plots/` có 8 PNG (4 components train_vs_test + 2 sales before_after)
- Sau khi chạy xong, báo kết quả cho assistant để viết tiếp `outputTask21_summary.md` (tiếng Việt, sẵn sàng paste vào paper Section 3.x).
